# 7. Teste oficial

Antes de pontuar o teste, execute novamente `notebooks/01_auditoria_inicial.ipynb` do início ao fim com o `final_treino_val.csv` completo. Isso refaz a validação aninhada de comparação e a busca final estratificada em todos os registros rotulados; em seguida, este notebook ajusta os pipelines com esses parâmetros em 100% dos rótulos.

O teste oficial é somente pontuado. Seus atributos e rótulos não participam do pré-processamento ajustado, treinamento, seleção de modelo ou hiperparâmetros, calibração ou escolha de limiar. Se existir `DESFECHO`, ele é consultado depois de gerar e salvar os scores, exclusivamente para métricas finais. Não use essas métricas para escolher entre candidatos se quiser preservar o teste como avaliação final.

Os scores por registro ficam em `results/official_test_model_scores.csv`, ignorado pelo Git. A saída não é necessariamente uma submissão pronta; confira o formato oficial.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display
from src.test_evaluation import evaluate_official_test

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "test_evaluation.py").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Abra este notebook dentro do repositório do projeto.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


print(f"Projeto: {project_root}")
print("Arquivo esperado: data/raw/final_teste.csv")

## Protocolo e proteção contra vazamento

O `id` fica fora dos preditores. A imputação, escala e codificação são ajustadas dentro de cada fold da busca final, por `Pipeline`/`ColumnTransformer`; após selecionar parâmetros com CV nos 100% de dados rotulados, o estimador é refitado em todos eles. O teste só recebe predições. A tabela final inclui AUC-ROC e, como métricas secundárias, sensibilidade, especificidade e F1 usando os limiares padrão (0,5 para probabilidades; 0 para `decision_function` do SVC). Esses limiares não são calibrados nem escolhidos com os rótulos do teste.

O arquivo de treino deve ser o mesmo usado na última execução do notebook 01. O hash SHA-256 registrado no artefato de busca detecta alterações. Se houver mudança, execute novamente a validação e a busca antes de pontuar.


In [ ]:
metrics_table, prediction_scores, roc_figure, test_summary = evaluate_official_test(
    project_root=project_root,
    train_filename="final_treino_val.csv",
    test_filename="final_teste.csv",
)

if test_summary["test_labels_available"]:
    print("Scores dos seis modelos gerados antes da avaliação final dos rótulos do teste. A tabela não deve ser usada para selecionar o modelo.")
    display(metrics_table)
    if roc_figure is not None:
        display(roc_figure)
else:
    print("O teste não contém DESFECHO: somente scores e classes pelos limiares padrão foram gerados.")
    display(metrics_table[["Modelo", "Tipo de score", "Limiar padrão"]])

print(f"Registros pontuados: {test_summary['test_rows_scored']:,}")
print("Métricas agregadas: results/official_test_metrics.json")
print("Scores por registro: results/official_test_model_scores.csv (local/ignorado pelo Git)")


## Como interpretar a saída

AUC-ROC, quando calculável, usa os scores contínuos e é a métrica principal; `decision_function` do SVC não é probabilidade calibrada. A tabela final inclui AUC-ROC para os seis modelos e, quando há rótulos, métricas com limiar padrão: 0,5 para probabilidades e 0 para o score do SVC. Sensibilidade, especificidade e F1 dependem desse limiar e são descritivas; a AUC-ROC permanece a métrica oficial. Não use os resultados do teste para escolher modelo ou ajustar limiar.
